Perform Sparse Memory Finetuning with a finetuning dataset made from misclassification of the validation set.

In [21]:
import os

if not os.path.exists('./models_dir'):
    os.makedirs('./models_dir')

Check if pre-trained memory augmented tiny ViT model is present.

In [28]:
weights_path = './models_dir/tiny_vit_memory_plus.pt'

# Check if the file exists
"The file exists." if os.path.exists(weights_path) else f"The file does not exist. Drag it into the {weights_path} path before continuing."

'The file exists.'

In [23]:
import sys
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import timm
import math
import torch.nn.functional as F
from tqdm import tqdm
import numpy as np
import gc

Declare the Memory Plus Layer Class.

In [24]:

class MemoryPlusLayer(nn.Module):

    def __init__(self, d_model, memory_slots, top_k = 32):
        # Define your memory mechanism here
        # Using Berges et al. (2024) "Memory Layers at scale" as a reference for the memory layer design

        super().__init__()

        self.key_dim = d_model // 2
        self.subkey_dim = self.key_dim // 2
        self.value_dim = d_model # <-- NOTE: May experiment with this value, as it may affect performance and memory usage.

        # Total memory_slots = |C| * |C'|. Sub-key matrices have sqrt(memory_slots) rows.
        self.num_subkeys = math.isqrt(memory_slots)
        assert self.num_subkeys ** 2 == memory_slots, f"memory_slots (n = {memory_slots}) must be a perfect square."


        # Query MLP
        self.query = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.SiLU(), # <-- Should match whatever the base models FFN activation function is.
            nn.Linear(d_model * 4, self.key_dim)
        )

        # Sub-Key Matrix One and Two
        # NOTE: Don't use nn.linear here, due to sparse key retrieval mechanism in forward pass.
        self.subkey_one = nn.Parameter(torch.empty(self.num_subkeys, self.subkey_dim, dtype=torch.float32))
        self.subkey_two = nn.Parameter(torch.empty(self.num_subkeys, self.subkey_dim, dtype=torch.float32))
        nn.init.uniform_(self.subkey_one, a = -1, b = 1)
        nn.init.uniform_(self.subkey_two, a = -1, b = 1)

        # Value Matrix
        self.values = nn.Parameter(torch.empty(memory_slots, self.value_dim, dtype=torch.float32))
        nn.init.normal_(self.values, std=0.02)  # apparently from lample et al 2019, CAN't FIND ITS REFERENCE

        # Weight Matrix One
        self.W1 = nn.Linear(d_model, self.value_dim, bias=False)

        # Weight Matrix Two
        self.W2 = nn.Linear(self.value_dim, d_model, bias=False)

        # Silu Activation Function
        self.silu = nn.SiLU()

        # QK-Normalisation,
        # NOTE:I think its more a general backbone design choice for memory layer, potentially place this after residual connection as we are using interleaved architecture (at end of this gated memory layer)
        """
        NOTE: This is a technique used to stabilize training and improve convergence in transformer models.
        """
        self.qk_norm = nn.RMSNorm(self.subkey_dim)

        # Top-K Selection
        """
        NOTE: Can experiment with this value, as it may affect performance and memory usage.
        """
        self.top_k = top_k

        # Softmax
        self.softmax = nn.Softmax(dim=-1)


    def lookup_memory(self, query):

        # 0. Get sub-queries by splitting query vector into halves
        q1, q2 = torch.chunk(query, 2, -1)

        # 1. Apply qk-normalisation for cosine similarity style lookup
        q1 = self.qk_norm(q1)
        q2 = self.qk_norm(q2)

        k1 = self.qk_norm(self.subkey_one)
        k2 = self.qk_norm(self.subkey_two)

        # 2. Get similarity subkey scores with query
        sim_scores_1 = q1 @ k1.T
        sim_scores_2  = q2 @ k2.T
        all_scores = sim_scores_1.unsqueeze(-1) + sim_scores_2.unsqueeze(-2)

        # 3. Cartesian Product Search:
        all_scores = all_scores.view(*all_scores.shape[:-2], -1)

        # 4. Select the final top-k combinations
        top_k_scores, top_k_indices = torch.topk(all_scores, self.top_k, dim=-1)

        # 5. Retrieve Values and Aggregate
        s = self.softmax(top_k_scores)

        # 6. Gather Values and Aggregate: NOTE: Using EmbeddingBag!
        # TODO: Make CUDA kernel to quicken EmbeddingBag solution
        flat_indices = top_k_indices.view(-1, self.top_k)
        flat_weights = s.view(-1, self.top_k)
        y_flat = F.embedding_bag(flat_indices, self.values, per_sample_weights=flat_weights, mode='sum')

        return y_flat.view(*query.shape[:-1], self.value_dim)

    def forward(self, x):

        q = self.query(x)

        y = self.lookup_memory(q)

        # TODO: Calculate Term-Frequency on batch x

        m_plus = self.silu(self.W1(x))
        m_plus = y * m_plus
        m_plus = self.W2(m_plus)

        return m_plus


Create the finetuning dataset and baseline stable set for measuring catastrophic forgetting.

In [25]:

def create_datasets(model, device):

    # Preprocess and prepare original dataset for tiny ViT

    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    data = datasets.FashionMNIST(root='./data_dir', train=False, download=True, transform=transform)
    loader = DataLoader(data, batch_size=64, shuffle=False)

    model.eval()
    misclassified_indices = []
    correct_indices = []

    print("Identifying hard examples in validation set...")
    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)

            # Determine which samples in the batch were wrong
            mask = predicted.ne(labels)

            # Convert batch-relative indices to global dataset indices
            batch_start = i * loader.batch_size
            for j, is_wrong in enumerate(mask):
                global_idx = batch_start + j
                if is_wrong:
                    misclassified_indices.append(global_idx)
                else:
                    correct_indices.append(global_idx)

    # Create the Subsets
    # Use the original dataset from the loader
    dataset = loader.dataset
    finetune_set = Subset(dataset, misclassified_indices)
    stable_set = Subset(dataset, correct_indices)

    print(f"Extraction complete:\n   {len(finetune_set)} misclassified samples found.\n   {len(stable_set)} classified samples found. ")

    return finetune_set, stable_set

Initialise the memory augmented tiny ViT model

In [29]:
def load_model():
  # Init Device
  device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

  # Set up the model architecture
  model = timm.create_model('vit_tiny_patch16_224', pretrained=False, num_classes=10, cache_dir = "./models_dir").to(device)
  d_model = model.embed_dim
  memory_slots = 256**2
  model.blocks[6].mlp = MemoryPlusLayer(d_model=d_model, memory_slots=memory_slots).to(device)

  # load weights:
  model.load_state_dict(torch.load(weights_path, weights_only=True))

  model.eval()

  return model, device

In [30]:
model, device = load_model()

RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory

In [ ]:
finetune_set, stable_set = create_datasets(model, device)